<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-07/blob/main/hw_07_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 — Time Series and Semi-Structured Data for ML

## 1. Setup and NYC Taxi Data Loading

In [69]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [70]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

In [71]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1.1 NYC Taxi Data Loading

Dataset: NYC TLC Yellow Taxi Trip Records, March 2024.
Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [72]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(URL, parquet_path)

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

df = pd.read_parquet(parquet_path, columns=columns)

df = df[
    (df["tpep_pickup_datetime"] >= "2024-03-01") &
    (df["tpep_pickup_datetime"] < "2024-04-01")
]

df = df.sample(
    n=min(500_000, len(df)),
    random_state=42
).copy()

print("Sampled date range:")
print(df["tpep_pickup_datetime"].min())
print(df["tpep_pickup_datetime"].max())

Sampled date range:
2024-03-01 00:00:03
2024-03-31 23:59:56


In [73]:
rows_before = len(df)

df = df[
    (df["fare_amount"] > 0) &
    (df["fare_amount"] < 500) &
    (df["trip_distance"] > 0) &
    (df["trip_distance"] < 100) &
    (df["PULocationID"].notna())
].copy()

print("Before cleaning:", rows_before)
print("After cleaning:", len(df))
print("Removed rows:", rows_before - len(df))

Before cleaning: 500000
After cleaning: 480094
Removed rows: 19906


In [74]:
df.to_sql(
    "nyc_taxi_raw",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
)

pd.read_sql("""
    SELECT COUNT(*) AS n_rows
    FROM nyc_taxi_raw;
""", engine)

,n_rows
0,480094


## 1.2 Daily Time-Series Aggregation

Aggregate NYC Taxi trips by pickup zone and local NYC calendar day.

In [75]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_daily;

        CREATE TABLE t7_hw_taxi_daily AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
            COUNT(*)::INTEGER AS trip_count,
            AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
            SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
        FROM nyc_taxi_raw
        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
          AND "PULocationID" IS NOT NULL
        GROUP BY
            "PULocationID"::INTEGER,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

        ALTER TABLE t7_hw_taxi_daily
            ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """))

print("Daily aggregation created successfully.")

Daily aggregation created successfully.


In [76]:
# Validation
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily;
""", engine))

display(pd.read_sql("""
    SELECT *
    FROM t7_hw_taxi_daily
    ORDER BY pickup_zone_id, ts_day
    LIMIT 5;
""", engine))

,n_rows,n_zones,first_day,last_day,total_trips
0,5438,248,2024-03-01,2024-03-31,480094


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance
0,1,2024-03-01,1,105.0,0.04
1,1,2024-03-02,2,95.0,18.43
2,1,2024-03-08,1,105.0,0.04
3,1,2024-03-11,1,10.0,2.07
4,1,2024-03-14,1,29.6,2.93


## 1.3 Calendar Spine — Complete Daily Time Series

Generate all calendar days for every observed pickup zone.
Missing daily trip counts are filled with zero.

In [77]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP VIEW IF EXISTS v_t7_hw_taxi_features;
        DROP TABLE IF EXISTS t7_hw_taxi_daily_spine;

        CREATE TABLE t7_hw_taxi_daily_spine AS
        WITH zones AS (
            SELECT DISTINCT pickup_zone_id
            FROM t7_hw_taxi_daily
        ),
        calendar AS (
            SELECT generate_series(
                DATE '2024-03-01',
                DATE '2024-03-31',
                INTERVAL '1 day'
            )::DATE AS ts_day
        )
        SELECT
            z.pickup_zone_id,
            c.ts_day,
            COALESCE(d.trip_count, 0)::INTEGER AS trip_count,
            d.avg_fare,
            COALESCE(d.total_distance, 0)::NUMERIC(12, 2) AS total_distance
        FROM zones z
        CROSS JOIN calendar c
        LEFT JOIN t7_hw_taxi_daily d
            ON d.pickup_zone_id = z.pickup_zone_id
           AND d.ts_day = c.ts_day;

        ALTER TABLE t7_hw_taxi_daily_spine
            ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """))

print("Calendar spine created successfully.")

Calendar spine created successfully.


In [78]:
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        COUNT(DISTINCT ts_day) AS n_days,
        COUNT(*) FILTER (WHERE trip_count = 0) AS zero_trip_days,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM t7_hw_taxi_daily_spine;
""", engine))

,n_rows,n_zones,n_days,zero_trip_days,first_day,last_day
0,7688,248,31,2250,2024-03-01,2024-03-31


In [79]:
display(pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily_spine
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 35;
""", engine))

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,NaN,NaN
1,1,2024-03-02,2,1.0,NaN,NaN
2,1,2024-03-03,0,2.0,NaN,NaN
3,1,2024-03-04,0,0.0,NaN,NaN
4,1,2024-03-05,0,0.0,NaN,NaN
5,1,2024-03-06,0,0.0,NaN,NaN
6,1,2024-03-07,0,0.0,NaN,NaN
7,1,2024-03-08,1,0.0,1.0,NaN
8,1,2024-03-09,0,1.0,2.0,NaN
9,1,2024-03-10,0,0.0,0.0,NaN


In [80]:
display(pd.read_sql("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 30) OVER (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ) AS trips_lag_30d
    FROM t7_hw_taxi_daily_spine
    WHERE pickup_zone_id = 1
    ORDER BY ts_day DESC
    LIMIT 3;
""", engine))

,pickup_zone_id,ts_day,trip_count,trips_lag_30d
0,1,2024-03-31,0,1.0
1,1,2024-03-30,0,NaN
2,1,2024-03-29,0,NaN


In [81]:
display(pd.read_sql("""
    WITH lagged AS (
        SELECT
            pickup_zone_id,
            ts_day,
            LAG(trip_count, 30) OVER (
                PARTITION BY pickup_zone_id
                ORDER BY ts_day
            ) AS trips_lag_30d
        FROM t7_hw_taxi_daily_spine
    )
    SELECT
        COUNT(*) AS total_rows,
        COUNT(trips_lag_30d) AS rows_with_lag_30d,
        COUNT(DISTINCT pickup_zone_id)
            FILTER (WHERE trips_lag_30d IS NOT NULL) AS zones_with_lag_30d
    FROM lagged;
""", engine))

,total_rows,rows_with_lag_30d,zones_with_lag_30d
0,7688,248,248


### Task 1 — Data Loading and Preparation

The NYC Taxi dataset (March 2024) was downloaded in Parquet format. A reproducible random sample of 500,000 trips was selected and cleaned by removing invalid fares, distances, and missing pickup zone IDs.

The cleaned data (480,094 trips) was loaded into PostgreSQL and aggregated by pickup zone and calendar day, producing 5,438 daily records across 248 zones.

A **Calendar Spine** was created using `generate_series()` and `CROSS JOIN` to ensure that every pickup zone has a continuous daily time series. Missing dates were filled with zero trip counts, while unknown average fares remained `NULL`.

The final calendar-aligned dataset contains **7,688 rows (248 zones × 31 days)** and supports accurate calendar-based lag, rolling, and expanding feature calculations.

**Note:** Since the dataset is sampled, zero trip counts represent missing observations in the sample and do not necessarily indicate zero actual taxi demand.

## 2. Time-Series ML Feature Engineering

### Combined Feature View
Lag, rolling, and expanding features calculated per pickup zone using a complete calendar spine.

In [82]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
        WITH features_basic AS (
            SELECT
                pickup_zone_id,
                ts_day,
                trip_count,
                avg_fare,
                total_distance,

                LAG(trip_count, 1) OVER w_pid AS trips_lag_1d,
                LAG(trip_count, 7) OVER w_pid AS trips_lag_7d,
                LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

                AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
                COUNT(avg_fare) OVER w7 AS fare_days_7d,

                STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
                COUNT(avg_fare) OVER w30 AS fare_days_30d,

                SUM(trip_count) OVER we AS cumulative_trips,
                AVG(trip_count) OVER we AS expanding_avg_trips

            FROM t7_hw_taxi_daily_spine

            WINDOW
                w_pid AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day
                ),
                w7 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP
                    RANGE BETWEEN INTERVAL '6 days' PRECEDING
                              AND CURRENT ROW
                ),
                w30 AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day::TIMESTAMP
                    RANGE BETWEEN INTERVAL '29 days' PRECEDING
                              AND CURRENT ROW
                ),
                we AS (
                    PARTITION BY pickup_zone_id
                    ORDER BY ts_day
                    ROWS BETWEEN UNBOUNDED PRECEDING
                             AND CURRENT ROW
                )
        )
        SELECT *
        FROM features_basic
        WHERE trips_lag_30d IS NOT NULL;
    """))

print("Feature view created successfully.")

Feature view created successfully.


In [83]:
#Validation
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        COUNT(*) FILTER (
            WHERE rolling_avg_fare_7d IS NOT NULL
        ) AS valid_rolling_avg,
        COUNT(*) FILTER (
            WHERE rolling_std_fare_30d IS NOT NULL
        ) AS valid_rolling_std
    FROM v_t7_hw_taxi_features;
""", engine))

display(pd.read_sql("""
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id
    LIMIT 10;
""", engine))

,n_rows,n_zones,first_day,last_day,valid_rolling_avg,valid_rolling_std
0,248,248,2024-03-31,2024-03-31,234,233


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,fare_days_7d,rolling_std_fare_30d,fare_days_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,NaN,0.00,0,1,1,NaN,0,36.149456,9,12,0.387097
1,2,2024-03-31,0,NaN,0.00,0,0,0,NaN,0,NaN,1,1,0.032258
2,3,2024-03-31,0,NaN,0.00,0,1,1,51.750000,4,13.525523,14,18,0.580645
3,4,2024-03-31,48,16.94,129.68,77,64,27,18.208571,7,2.161566,30,1004,32.387097
4,6,2024-03-31,0,NaN,0.00,0,0,0,85.600000,1,49.415410,3,3,0.096774
5,7,2024-03-31,18,22.59,89.95,15,14,5,22.465714,7,5.049312,30,298,9.612903
6,8,2024-03-31,0,NaN,0.00,0,0,0,NaN,0,NaN,1,1,0.032258
7,9,2024-03-31,0,NaN,0.00,1,1,0,14.255000,2,20.396997,7,7,0.225806
8,10,2024-03-31,5,75.85,67.30,2,5,12,58.257143,7,12.322941,30,162,5.225806
9,11,2024-03-31,0,NaN,0.00,0,0,0,37.000000,2,11.738678,10,15,0.483871


### Task 2 — Time-Series Feature Engineering Summary

Time-series ML features were generated using PostgreSQL window functions, partitioned by pickup zone. The features include 1-, 7-, and 30-day lags, rolling 7-day average fare, rolling 30-day fare volatility, and expanding cumulative trip statistics.

A calendar spine ensures consistent daily intervals, making lag features correspond to actual calendar days. The final feature view contains **248 records**, one per pickup zone for March 31, 2024.

Rolling average fares are available for 234 zones, while rolling volatility is available for 233 zones. Missing values occur when insufficient fare observations exist within the corresponding window.

For production forecasting, additional historical data and leakage-safe feature alignment would be required.

## 3. JSONB Event Log

Create a synthetic event log with five event types and heterogeneous JSONB payloads.

In [84]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_events;

        CREATE TABLE t7_hw_events (
            event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
            user_id     INTEGER NOT NULL,
            event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
            event_type  TEXT NOT NULL,
            payload     JSONB NOT NULL
        );

        INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
        SELECT
            (n % 1000) + 1 AS user_id,
            TIMESTAMPTZ '2024-04-01 00:00:00+00'
                - ((n % 60) || ' days')::INTERVAL AS event_ts,

            CASE (n % 5)
                WHEN 0 THEN 'page_view'
                WHEN 1 THEN 'click'
                WHEN 2 THEN 'add_to_cart'
                WHEN 3 THEN 'checkout'
                ELSE 'recommendation_shown'
            END AS event_type,

            CASE (n % 5)
                WHEN 0 THEN jsonb_build_object(
                    'url', '/product/' || ((n % 200) + 1),
                    'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
                    'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
                )

                WHEN 1 THEN jsonb_build_object(
                    'element_id', 'btn_' || ((n % 50) + 1),
                    'page_url', '/category/' || ((n % 20) + 1),
                    'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
                )

                WHEN 2 THEN jsonb_build_object(
                    'product_id', (n % 200) + 1,
                    'price', ((n % 100) + 10)::NUMERIC(10, 2),
                    'quantity', (n % 5) + 1,
                    'tags', jsonb_build_array(
                        (ARRAY['promo','new','clearance'])[(n % 3) + 1]
                    )
                )

                WHEN 3 THEN jsonb_build_object(
                    'order_id', 'o_' || (n + 100000),
                    'total', ((n % 500) + 50)::NUMERIC(10, 2),
                    'items_count', (n % 10) + 1,
                    'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
                )

                ELSE jsonb_build_object(
                    'model_name', 'rec_v' || ((n % 5) + 1),
                    'model_version', '1.' || (n % 10),
                    'recommendations', jsonb_build_array(
                        (n % 200) + 1,
                        ((n + 1) % 200) + 1,
                        ((n + 2) % 200) + 1
                    ),
                    'experiment', 'exp_' || ((n % 3) + 1)
                )
            END AS payload

        FROM generate_series(1, 50000) AS n;

        ANALYZE t7_hw_events;
    """))

print("JSONB event log created successfully.")

JSONB event log created successfully.


In [85]:
display(pd.read_sql("""
    SELECT
        event_type,
        COUNT(*) AS n_events
    FROM t7_hw_events
    GROUP BY event_type
    ORDER BY event_type;
""", engine))

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [86]:
display(pd.read_sql("""
    SELECT DISTINCT ON (event_type)
        event_type,
        payload
    FROM t7_hw_events
    ORDER BY event_type, event_id;
""", engine))

,event_type,payload
0,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
2,click,"{'device': 'desktop', 'page_url': '/category/2..."
3,page_view,"{'url': '/product/6', 'device': 'tablet', 'ref..."
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


In [87]:
display(pd.read_sql("""
    SELECT
        event_id,
        user_id,
        event_ts,
        payload
    FROM t7_hw_events
    WHERE event_type = 'checkout'
      AND payload @> '{"payment_method": "card"}'::jsonb
    LIMIT 10;
""", engine))

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


In [88]:
# Checking events `add_to_cart` , in which we have `tags`
display(pd.read_sql("""
    SELECT
        event_id,
        event_type,
        payload
    FROM t7_hw_events
    WHERE payload ? 'tags'
    LIMIT 10;
""", engine))

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [89]:
display(pd.read_sql("""
    SELECT
        event_id,
        event_type,
        payload
    FROM t7_hw_events
    WHERE payload ?| ARRAY['experiment', 'model_version']
    LIMIT 10;
""", engine))

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


In [90]:
# Retrieving recommended products
display(pd.read_sql("""
    SELECT
        event_id,
        jsonb_path_query(
            payload,
            '$.recommendations[*]'
        ) AS recommended_pid
    FROM t7_hw_events
    WHERE event_type = 'recommendation_shown'
    LIMIT 15;
""", engine))

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [91]:
#Checkout with amount more 300
display(pd.read_sql("""
    SELECT
        event_id,
        payload ->> 'order_id' AS order_id,
        (payload ->> 'total')::NUMERIC(10, 2) AS total
    FROM t7_hw_events
    WHERE event_type = 'checkout'
      AND jsonb_path_exists(
          payload,
          '$.total ? (@ > 300)'
      )
    LIMIT 10;
""", engine))

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


### Task 3 — JSONB Event Log Summary

A synthetic event log was created in PostgreSQL with 50,000 records across five event types: `page_view`, `click`, `add_to_cart`, `checkout`, and `recommendation_shown`.

The `payload` column uses JSONB to store heterogeneous event attributes without requiring a separate table schema for each event type.

Queries demonstrate JSONB containment (`@>`), key existence (`?`, `?|`), and JSONPath operations for extracting recommendations and filtering checkout amounts.

This approach is useful for flexible event ingestion, experimentation tracking, recommendation systems, and user behavior analytics.

## 4. JSONB Index Performance Analysis

### 4.1 Baseline — Containment Query Without GIN Index

In [92]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
        DROP INDEX IF EXISTS idx_t7_events_device_btree;

        ANALYZE t7_hw_events;
    """))

In [93]:
plan = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object('payment_method', 'card');
""", engine)

print("\n".join(plan["QUERY PLAN"]))

Aggregate  (cost=1758.99..1759.00 rows=1 width=8) (actual time=63.552..63.553 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3598 width=0) (actual time=0.021..63.202 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.202 ms
Execution Time: 63.582 ms


### 4.2 GIN Index for JSONB Containment

Create a GIN index using `jsonb_path_ops` to optimize JSONB containment queries.

In [94]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_payload_gin_pathops
            ON t7_hw_events USING GIN (payload jsonb_path_ops);

        ANALYZE t7_hw_events;
    """))

print("GIN index created successfully.")

GIN index created successfully.


In [95]:
#Check index size
display(pd.read_sql("""
    SELECT
        pg_size_pretty(
            pg_relation_size('idx_t7_events_payload_gin_pathops')
        ) AS gin_index_size;
""", engine))

,gin_index_size
0,1144 kB


In [96]:
plan = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object('payment_method', 'card');
""", engine)

print("\n".join(plan["QUERY PLAN"]))

Aggregate  (cost=1084.20..1084.21 rows=1 width=8) (actual time=5.897..5.899 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.47..1076.81 rows=2956 width=0) (actual time=0.797..5.678 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.73 rows=2956 width=0) (actual time=0.549..0.550 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=17
Planning Time: 0.373 ms
Execution Time: 5.938 ms


### GIN Index — Performance Comparison

The JSONB containment query initially used a sequential scan and took **407.777 ms** to identify 3,334 matching events out of 50,000 records.

After creating a GIN index with `jsonb_path_ops`, PostgreSQL switched to `Bitmap Index Scan` and `Bitmap Heap Scan`, reducing execution time to **7.151 ms** (approximately **57× faster** in this test).

The estimated total cost decreased from 1761.48 to 1061.12. The GIN index occupies 1,144 kB of additional storage.

This demonstrates how GIN can accelerate selective JSONB containment queries, although actual performance depends on data distribution, caching, and query selectivity.

In [97]:
plan = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)

print("\n".join(plan["QUERY PLAN"]))

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=11.079..11.080 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.019..10.585 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.085 ms
Execution Time: 11.109 ms


In [98]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree
            ON t7_hw_events ((payload ->> 'device'));

        ANALYZE t7_hw_events;
    """))

print("Expression B-tree index created successfully.")

Expression B-tree index created successfully.


In [99]:
plan = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload ->> 'device' = 'mobile';
""", engine)

print("\n".join(plan["QUERY PLAN"]))

Aggregate  (cost=1243.18..1243.19 rows=1 width=8) (actual time=3.871..3.873 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.02..1222.30 rows=8352 width=0) (actual time=0.828..3.252 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.93 rows=8352 width=0) (actual time=0.632..0.632 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.269 ms
Execution Time: 3.903 ms


### Task 4 — JSONB Index Performance Summary

Two indexing strategies were evaluated using `EXPLAIN (ANALYZE, BUFFERS)` on 50,000 synthetic events.

The JSONB containment query initially used a sequential scan and took **407.777 ms**. After creating a GIN index with `jsonb_path_ops`, PostgreSQL used `Bitmap Index Scan` and `Bitmap Heap Scan`, reducing execution time to **7.151 ms**, approximately 57 times faster.

For the expression `payload ->> 'device' = 'mobile'`, an expression B-tree index reduced execution time from **51.692 ms** to **3.517 ms**, approximately 14.7 times faster.

GIN is suitable for searching JSONB structures, whereas expression B-tree indexes are effective for frequently queried scalar values extracted from JSONB.

Both indexes improve read performance but require additional storage and introduce maintenance overhead during inserts and updates. Actual performance depends on query selectivity, data distribution, and caching.

## 5. JSONB vs Normalized Schema

### 5.1 Create a Normalized Checkout Table

Extract structured checkout attributes from JSONB into typed SQL columns.

In [100]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_checkouts_norm;

        CREATE TABLE t7_hw_checkouts_norm (
            event_id        INTEGER PRIMARY KEY,
            user_id         INTEGER NOT NULL,
            event_ts        TIMESTAMPTZ NOT NULL,
            order_id        TEXT NOT NULL,
            total           NUMERIC(10, 2) NOT NULL,
            items_count     INTEGER NOT NULL CHECK (items_count > 0),
            payment_method  TEXT NOT NULL
                CHECK (payment_method IN ('card', 'paypal', 'crypto'))
        );

        INSERT INTO t7_hw_checkouts_norm (
            event_id,
            user_id,
            event_ts,
            order_id,
            total,
            items_count,
            payment_method
        )
        SELECT
            event_id,
            user_id,
            event_ts,
            payload ->> 'order_id',
            (payload ->> 'total')::NUMERIC(10, 2),
            (payload ->> 'items_count')::INTEGER,
            payload ->> 'payment_method'
        FROM t7_hw_events
        WHERE event_type = 'checkout';

        CREATE INDEX idx_t7_checkouts_norm_pm
            ON t7_hw_checkouts_norm (payment_method);

        ANALYZE t7_hw_checkouts_norm;
    """))

print("Normalized checkout table created successfully.")

Normalized checkout table created successfully.


In [101]:
#Validation
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_checkouts,
        COUNT(DISTINCT payment_method) AS n_payment_methods,
        MIN(total) AS min_total,
        MAX(total) AS max_total,
        AVG(total)::NUMERIC(10, 2) AS avg_total
    FROM t7_hw_checkouts_norm;
""", engine))

,n_checkouts,n_payment_methods,min_total,max_total,avg_total
0,10000,3,53.0,548.0,300.5


In [102]:
plan_jsonb = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT
        payload ->> 'payment_method' AS payment_method,
        AVG((payload ->> 'total')::NUMERIC) AS avg_total,
        COUNT(*) AS n
    FROM t7_hw_events
    WHERE event_type = 'checkout'
    GROUP BY payload ->> 'payment_method'
    ORDER BY avg_total DESC
    LIMIT 3;
""", engine)

print("\n".join(plan_jsonb["QUERY PLAN"]))

Limit  (cost=1959.78..1959.78 rows=3 width=72) (actual time=14.418..14.420 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1959.78..1973.86 rows=5632 width=72) (actual time=14.417..14.418 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1802.50..1886.98 rows=5632 width=72) (actual time=14.376..14.407 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.36 rows=10143 width=132) (actual time=0.016..9.156 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=10
Planning Time: 0.135 ms
Execution Time: 14.505 ms


In [103]:
plan_normalized = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT
        payment_method,
        AVG(total) AS avg_total,
        COUNT(*) AS n
    FROM t7_hw_checkouts_norm
    GROUP BY payment_method
    ORDER BY avg_total DESC
    LIMIT 3;
""", engine)

print("\n".join(plan_normalized["QUERY PLAN"]))

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=4.369..4.371 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=4.368..4.369 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=4.355..4.357 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.013..1.102 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.100 ms
Execution Time: 4.416 ms


In [104]:
comparison = pd.read_sql("""
    WITH jsonb_result AS (
        SELECT
            payload ->> 'payment_method' AS payment_method,
            AVG((payload ->> 'total')::NUMERIC) AS avg_total,
            COUNT(*) AS n
        FROM t7_hw_events
        WHERE event_type = 'checkout'
        GROUP BY payload ->> 'payment_method'
    ),
    normalized_result AS (
        SELECT
            payment_method,
            AVG(total) AS avg_total,
            COUNT(*) AS n
        FROM t7_hw_checkouts_norm
        GROUP BY payment_method
    )
    SELECT
        j.payment_method,
        ROUND(j.avg_total, 2) AS jsonb_avg,
        ROUND(n.avg_total, 2) AS normalized_avg,
        j.n AS jsonb_count,
        n.n AS normalized_count,
        ROUND(j.avg_total, 2) = ROUND(n.avg_total, 2)
            AND j.n = n.n AS results_match
    FROM jsonb_result j
    JOIN normalized_result n
        ON n.payment_method = j.payment_method
    ORDER BY j.payment_method;
""", engine)

display(comparison)

,payment_method,jsonb_avg,normalized_avg,jsonb_count,normalized_count,results_match
0,card,300.50,300.50,3334,3334,True
1,crypto,300.48,300.48,3333,3333,True
2,paypal,300.52,300.52,3333,3333,True


## Summary

This homework demonstrated how PostgreSQL can support flexible JSONB event storage while using different indexing strategies to improve query performance.

Before creating the GIN index, the JSONB containment query used a `Seq Scan` and took **407.777 ms**, scanning 50,000 events to identify 3,334 matching records. After adding a GIN index with `jsonb_path_ops`, PostgreSQL switched to `Bitmap Index Scan` and `Bitmap Heap Scan`, reducing execution time to **7.151 ms**. The estimated total cost decreased from 1761.48 to 1061.12.

GIN indexes are effective for JSONB containment (`@>`) because they index searchable JSON structures. However, queries such as `payload ->> 'device' = 'mobile'` compare extracted text values. An expression B-tree index is more appropriate for this pattern. In our experiment, it reduced execution time from **51.692 ms to 3.517 ms**.

These improvements involve trade-offs. Indexes accelerate reads but consume additional storage and increase maintenance costs during inserts and updates. The GIN index required approximately **1,144 kB** of storage.

For ML event ingestion, I would choose JSONB when event types have different or evolving attributes. Normalized tables are preferable for frequently accessed, structured features requiring strict validation and efficient aggregation. Our normalized checkout query completed in **7.373 ms**, compared with **15.476 ms** for JSONB, while producing identical results.

Adding a new `refund` event type to JSONB would mainly require defining and validating a new payload structure. In a normalized approach, I would create a dedicated refund table with typed columns, constraints, and appropriate indexes. This requires additional schema migration work but provides stronger data integrity.

Overall, PostgreSQL JSONB offers flexibility for event ingestion, while normalized structures are better suited to curated analytical and ML workloads.